# Sentiment Analysis with HuggingFace Transformers

Fine-tuning a pre-trained transformer model for text classification.

1. **HuggingFace Ecosystem** - Models, tokenizers, datasets, Trainer API
2. **Tokenization** - WordPiece, subword tokens, special tokens
3. **Fine-tuning BERT** - Transfer learning for NLP
4. **Evaluation** - Metrics and error analysis

**Dataset**: IMDB Movie Reviews (binary sentiment)

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Dataset and Tokenizer

In [ ]:
# Load IMDB dataset
dataset = load_dataset("imdb")
print(dataset)
print(f"\nSample: {dataset['train'][0]['text'][:200]}...")
print(f"Label: {dataset['train'][0]['label']} (0=negative, 1=positive)")

In [ ]:
# Use a small subset for demonstration (full fine-tuning needs GPU)
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

# Load DistilBERT (smaller, faster than BERT, 97% of performance)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Tokenization example
sample_text = "This movie was absolutely fantastic! The acting was superb."
tokens = tokenizer(sample_text, return_tensors="pt")

print(f"Original: {sample_text}")
print(f"Token IDs: {tokens['input_ids'][0].tolist()}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"Attention mask: {tokens['attention_mask'][0].tolist()}")

## 2. Tokenize the Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )

train_tokenized = small_train.map(tokenize_function, batched=True)
test_tokenized = small_test.map(tokenize_function, batched=True)

# Set format for PyTorch
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

## 3. Fine-tune with HuggingFace Trainer

In [ ]:
# Load pre-trained model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none",  # Disable wandb/mlflow logging for demo
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train!
trainer.train()

In [ ]:
# Evaluate
results = trainer.evaluate()
print(f"\nTest Results:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

## 4. Inference

In [ ]:
# Predict on new text
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = probs.argmax().item()
    return {"label": "positive" if pred == 1 else "negative", "confidence": probs[0][pred].item()}

test_texts = [
    "This movie was absolutely brilliant! One of the best I've ever seen.",
    "Terrible film. Complete waste of time, awful acting.",
    "It was okay, nothing special but not bad either.",
]

for text in test_texts:
    result = predict_sentiment(text)
    print(f"\n{text}")
    print(f"  -> {result['label']} ({result['confidence']:.2%})")

## Key Takeaways

1. **HuggingFace makes fine-tuning easy** - Trainer API handles training loop, evaluation, checkpointing
2. **DistilBERT** is 40% smaller and 60% faster than BERT with 97% of performance
3. **Low learning rate (2e-5)** is critical for fine-tuning - higher rates destroy pre-trained weights
4. **Even 2000 samples can work** with transfer learning - pre-trained models encode enormous language knowledge
5. **Tokenizers matter** - WordPiece handles unknown words by splitting into subwords
6. **For production**: save with `model.save_pretrained()` and serve with TorchServe or FastAPI